In [21]:
import json
import random
import shutil
import unicodedata
from pathlib import Path
from collections import Counter
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_FOLDS = 5
QUARANTINE_IDS = {103, 104, 105, 108, 116, 121, 136, 141, 457}

DATA_PATH = Path(
    "/kaggle/input/datasets/tuandatdao/vgec2026-bartpho/VGEC 2026_v2.json"
)
SAVE_DIR = Path("/kaggle/working/vgec_preprocessed")

for folder in ["raw", "processed", "quarantine", "folds"]:
    (SAVE_DIR / folder).mkdir(parents=True, exist_ok=True)

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def normalize(text):
    # Chỉ chuẩn hóa Unicode, giữ cách viết, dấu câu và khoảng trắng.
    return unicodedata.normalize("NFC", text)

with open(DATA_PATH, "r", encoding="utf-8-sig") as f:
    raw_data = json.load(f)

# 1. Lưu nguyên bản raw.
shutil.copy2(DATA_PATH, SAVE_DIR / "raw" / DATA_PATH.name)

# 2. Tách 9 ID khỏi cả train và validation.
quarantine = [
    row for row in raw_data
    if row["ID"] in QUARANTINE_IDS
]
save_json(SAVE_DIR / "quarantine" / "records.json", quarantine)

# 3. Normalize, tạo identity pairs và gộp input trùng.
groups = {}

for row in raw_data:
    if row["ID"] in QUARANTINE_IDS:
        continue

    source = normalize(row["Input"])
    label = row["Output"]["Error_Type_Id"]

    if label == 0:
        references = [source]
    else:
        references = [
            normalize(ref)
            for ref in row["Output"]["Fixed Sentence"]
        ]

    if source not in groups:
        groups[source] = {
            "id": row["ID"],
            "original_ids": [],
            "source": source,
            "error_type": label,
            "references": []
        }

    sample = groups[source]
    assert sample["error_type"] == label, (
        f"Input trùng nhưng khác nhãn: {sample['original_ids']} và {row['ID']}"
    )

    sample["original_ids"].append(row["ID"])

    for ref in references:
        if ref not in sample["references"]:
            sample["references"].append(ref)

data = sorted(groups.values(), key=lambda sample: sample["id"])

# 4. Chia stratified 5-fold.
# Mỗi input đã được gộp thành đúng một sample,
# nên split ở đây cũng bảo đảm các input trùng không qua hai fold.
splitter = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

labels = [sample["error_type"] for sample in data]

for fold, (_, val_indices) in enumerate(
    splitter.split(data, labels)
):
    for index in val_indices:
        data[index]["fold"] = fold

save_json(SAVE_DIR / "processed" / "all.json", data)

# 5. Xuất train/validation cho từng fold.
fold_stats = []

for fold in range(N_FOLDS):
    fold_dir = SAVE_DIR / "folds" / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_data = [sample for sample in data if sample["fold"] != fold]
    val_data = [sample for sample in data if sample["fold"] == fold]

    train_sources = {sample["source"] for sample in train_data}
    val_sources = {sample["source"] for sample in val_data}
    assert train_sources.isdisjoint(val_sources)

    save_json(fold_dir / "train.json", train_data)
    save_json(fold_dir / "val.json", val_data)

    fold_stats.append({
        "fold": fold,
        "train_size": len(train_data),
        "val_size": len(val_data),
        "train_labels": dict(Counter(
            sample["error_type"] for sample in train_data
        )),
        "val_labels": dict(Counter(
            sample["error_type"] for sample in val_data
        ))
    })

# 6. Lưu thông tin xử lý và mapping truy ngược ID gốc.
metadata = {
    "seed": SEED,
    "n_folds": N_FOLDS,
    "normalization": "Unicode NFC only",
    "raw_size": len(raw_data),
    "quarantine_size": len(quarantine),
    "quarantine_ids": sorted(QUARANTINE_IDS),
    "duplicates_merged": len(raw_data) - len(quarantine) - len(data),
    "processed_size": len(data),
    "label_counts": dict(Counter(
        sample["error_type"] for sample in data
    )),
    "folds": fold_stats
}

save_json(SAVE_DIR / "processed" / "metadata.json", metadata)

save_json(
    SAVE_DIR / "processed" / "id_mapping.json",
    [
        {
            "id": sample["id"],
            "original_ids": sample["original_ids"],
            "fold": sample["fold"]
        }
        for sample in data
    ]
)

# 7. Đóng gói để tải về.
zip_path = shutil.make_archive(
    str(SAVE_DIR),
    "zip",
    root_dir=SAVE_DIR
)

print("Số mẫu raw:", len(raw_data))
print("Số mẫu quarantine:", len(quarantine))
print("Số bản ghi trùng được gộp:", metadata["duplicates_merged"])
print("Số mẫu sau preprocessing:", len(data))
print("Phân bố nhãn:", metadata["label_counts"])

for stats in fold_stats:
    print(
        f"Fold {stats['fold']}: "
        f"train={stats['train_size']}, "
        f"val={stats['val_size']} | "
        f"Nhãn val: {stats['val_labels']}"
    )

print("\nĐã lưu:", SAVE_DIR)
print("File ZIP:", zip_path)

Số mẫu raw: 457
Số mẫu quarantine: 9
Số bản ghi trùng được gộp: 7
Số mẫu sau preprocessing: 441
Phân bố nhãn: {1: 146, 2: 28, 3: 31, 4: 36, 0: 200}
Fold 0: train=352, val=89 | Nhãn val: {1: 30, 2: 5, 3: 6, 4: 8, 0: 40}
Fold 1: train=353, val=88 | Nhãn val: {1: 29, 2: 6, 3: 6, 4: 7, 0: 40}
Fold 2: train=353, val=88 | Nhãn val: {1: 29, 2: 6, 3: 6, 4: 7, 0: 40}
Fold 3: train=353, val=88 | Nhãn val: {1: 29, 2: 6, 3: 6, 4: 7, 0: 40}
Fold 4: train=353, val=88 | Nhãn val: {1: 29, 2: 5, 3: 7, 4: 7, 0: 40}

Đã lưu: /kaggle/working/vgec_preprocessed
File ZIP: /kaggle/working/vgec_preprocessed.zip


In [22]:
%pip install -q "transformers==4.57.1" "accelerate==1.10.1" sentencepiece protobuf

Note: you may need to restart the kernel to use updated packages.


In [23]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import math
import random
from pathlib import Path

import torch
import transformers
from packaging.version import Version
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback,
    set_seed,
)

assert torch.cuda.is_available(), "M cần bật GPU trước."
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), (
    "Checkpoint gốc dùng .bin và cần PyTorch >= 2.6 với Transformers này. "
    "Gửi t phiên bản torch đang dùng để chỉnh môi trường."
)

MODEL_NAME = "vinai/bartpho-syllable-base"
DATA_DIR = Path("/kaggle/working/vgec_preprocessed")
OUTPUT_DIR = Path("/kaggle/working/bartpho_vgec")

SEED = 42
EPOCHS = 15
LEARNING_RATE = 3e-5
TRAIN_BATCH_SIZE = 2
GRAD_ACCUMULATION = 8
EVAL_BATCH_SIZE = 2

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def write_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

GPU: Tesla T4
PyTorch: 2.11.0+cu128
Transformers: 5.16.1


In [24]:
from torch.utils.data import Dataset
from transformers import TrainerCallback
class CorrectionDataset(Dataset):
    def __init__(self, records, tokenizer, training=False):
        self.records = records
        self.training = training
        self.epoch = 0

        # Tokenize trước toàn bộ input và các reference.
        self.sources = [
            tokenizer(row["source"], truncation=False)
            for row in records
        ]

        self.targets = [
            [
                tokenizer(
                    text_target=ref,
                    truncation=False
                )["input_ids"]
                for ref in row["references"]
            ]
            for row in records
        ]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        refs = self.targets[index]

        if self.training:
            rng = random.Random(
                SEED + self.epoch * 1_000_003 + index
            )
            target = rng.choice(refs)
        else:
            target = refs[0]

        return {
            "input_ids": list(self.sources[index]["input_ids"]),
            "attention_mask": list(
                self.sources[index]["attention_mask"]
            ),
            "labels": list(target),
        }


class ReferenceEpochCallback(TrainerCallback):
    def __init__(self, dataset):
        self.dataset = dataset

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.dataset.epoch = int(state.epoch or 0)

In [25]:
torch.inference_mode()
def predict_records(model, records, batch_size=2, max_length=512):
    model.eval()
    device = next(model.parameters()).device
    predictions = []

    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]

        inputs = tokenizer(
            [row["source"] for row in batch],
            return_tensors="pt",
            padding=True,
            truncation=False,
        ).to(device)

        generated = model.generate(
            **inputs,
            num_beams=4,
            do_sample=False,
            max_length=max_length,
            early_stopping=True,
            use_cache=True,
        )

        predictions.extend(
            tokenizer.batch_decode(
                generated,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )
        )

    return predictions


def summarize_predictions(records, predictions):
    def score(rows):
        if not rows:
            return {}

        return {
            "n": len(rows),
            "exact_match": sum(
                pred in row["references"]
                for row, pred in rows
            ) / len(rows),
            "unchanged_rate": sum(
                pred == row["source"]
                for row, pred in rows
            ) / len(rows),
            "keep_input_exact_match": sum(
                row["source"] in row["references"]
                for row, _ in rows
            ) / len(rows),
        }

    pairs = list(zip(records, predictions))

    return {
        "overall": score(pairs),
        "by_error_type": {
            str(label): score([
                pair for pair in pairs
                if pair[0]["error_type"] == label
            ])
            for label in range(5)
        },
    }

In [34]:
import gc
import json
import math
import torch
import transformers

from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

def train_fold(fold):
    set_seed(SEED)
    gc.collect()
    torch.cuda.empty_cache()

    fold_dir = DATA_DIR / "folds" / f"fold_{fold}"
    run_dir = OUTPUT_DIR / f"fold_{fold}"
    run_dir.mkdir(parents=True, exist_ok=True)

    train_records = read_json(fold_dir / "train.json")
    val_records = read_json(fold_dir / "val.json")

    assert {
        row["source"] for row in train_records
    }.isdisjoint({
        row["source"] for row in val_records
    }), "Input bị trùng giữa train và validation."

    train_dataset = CorrectionDataset(
        train_records,
        tokenizer,
        training=True,
    )
    val_dataset = CorrectionDataset(
        val_records,
        tokenizer,
        training=False,
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

    # Kiểm tra độ dài để không âm thầm cắt input hoặc reference.
    lengths = []

    for dataset in [train_dataset, val_dataset]:
        lengths.extend(
            len(source["input_ids"])
            for source in dataset.sources
        )
        lengths.extend(
            len(target)
            for refs in dataset.targets
            for target in refs
        )

    longest = max(lengths)
    model_limit = model.config.max_position_embeddings

    assert longest <= model_limit, (
        f"Có mẫu dài {longest} tokens, "
        f"vượt giới hạn {model_limit}."
    )

    generation_max_length = min(
        model_limit,
        max(256, math.ceil((longest + 32) / 8) * 8),
    )

    model.config.use_cache = False

    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        pad_to_multiple_of=8,
        label_pad_token_id=-100,
    )

    # Warmup khoảng 10% tổng số optimizer steps.
    n_gpu = max(1, torch.cuda.device_count())
    batches_per_epoch = math.ceil(
        len(train_records) / (TRAIN_BATCH_SIZE * n_gpu)
    )
    steps_per_epoch = math.ceil(
        batches_per_epoch / GRAD_ACCUMULATION
    )
    warmup_steps = max(
        1,
        round(0.1 * EPOCHS * steps_per_epoch),
    )

    args = Seq2SeqTrainingArguments(
        output_dir=str(run_dir / "checkpoints"),
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION,

        weight_decay=0.01,
        warmup_steps=warmup_steps,
        lr_scheduler_type="linear",
        max_grad_norm=1.0,

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=5,
        logging_first_step=True,
        disable_tqdm=True,

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
    

        fp16=False,
        bf16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        },

        predict_with_generate=False,
        prediction_loss_only=True,
        dataloader_num_workers=0,
        report_to="none",
        optim="adamw_torch",
        seed=SEED,
        data_seed=SEED,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=collator,
        callbacks=[
            ReferenceEpochCallback(train_dataset),
            EarlyStoppingCallback(
                early_stopping_patience=3
            ),
        ],
    )

    print(f"\n===== FOLD {fold} =====", flush=True)
    print("Train:", len(train_records), flush=True)
    print("Validation:", len(val_records), flush=True)
    print("Độ dài lớn nhất:", longest, "tokens", flush=True)
    print("Warmup steps:", warmup_steps, flush=True)
    print("Bắt đầu training...", flush=True)

    trainer.train()

    # Trainer đã nạp lại checkpoint có validation loss thấp nhất.
    val_loss_metrics = trainer.evaluate()

    model = trainer.model
    model.gradient_checkpointing_disable()
    model.config.use_cache = True

    best_dir = run_dir / "best_model"
    trainer.save_model(str(best_dir))
    tokenizer.save_pretrained(str(best_dir))

    print("Đang sinh câu trên validation...", flush=True)

    predictions = predict_records(
        model,
        val_records,
        batch_size=EVAL_BATCH_SIZE,
        max_length=generation_max_length,
    )

    metrics = summarize_predictions(
        val_records,
        predictions,
    )
    metrics["fold"] = fold
    metrics["eval_loss"] = val_loss_metrics["eval_loss"]
    metrics["best_checkpoint"] = (
        trainer.state.best_model_checkpoint
    )

    prediction_rows = [
        {
            "id": row["id"],
            "original_ids": row["original_ids"],
            "source": row["source"],
            "error_type": row["error_type"],
            "references": row["references"],
            "prediction": prediction,
        }
        for row, prediction in zip(
            val_records,
            predictions,
        )
    ]

    write_json(
        run_dir / "val_predictions.json",
        prediction_rows,
    )
    write_json(run_dir / "metrics.json", metrics)
    write_json(
        run_dir / "training_log.json",
        trainer.state.log_history,
    )
    write_json(run_dir / "run_config.json", {
        "model": MODEL_NAME,
        "fold": fold,
        "seed": SEED,
        "epochs_max": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "train_batch_size_per_device": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUMULATION,
        "warmup_steps": warmup_steps,
        "generation_max_length": generation_max_length,
        "num_beams": 4,
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "validation_loss_reference": "first reference",
    })

    print(json.dumps(
        metrics,
        ensure_ascii=False,
        indent=2,
    ))
    print("Model đã lưu:", best_dir)

    del trainer, model, collator
    gc.collect()
    torch.cuda.empty_cache()

    return metrics

In [35]:
# Chạy fold 0 trước.
FOLDS = [0]

# Muốn chạy đủ 5 folds thì thay dòng trên bằng:
# FOLDS = list(range(5))

results = []

for fold in FOLDS:
    results.append(train_fold(fold))

write_json(OUTPUT_DIR / "results.json", results)

Loading weights:   0%|          | 0/265 [00:00<?, ?it/s]


===== FOLD 0 =====
Train: 352
Validation: 89
Độ dài lớn nhất: 64 tokens
Warmup steps: 16
Bắt đầu training...


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'loss': '14.11', 'grad_norm': '167.3', 'learning_rate': '0', 'epoch': '0.09091'}
{'loss': '15.99', 'grad_norm': '147.3', 'learning_rate': '7.5e-06', 'epoch': '0.4545'}
{'loss': '12.63', 'grad_norm': '111.5', 'learning_rate': '1.688e-05', 'epoch': '0.9091'}
{'eval_loss': '1.368', 'eval_runtime': '1.732', 'eval_samples_per_second': '51.38', 'eval_steps_per_second': '13.28', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

{'train_runtime': '150.9', 'train_samples_per_second': '34.99', 'train_steps_per_second': '1.094', 'train_loss': '7.766', 'epoch': '6'}


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': '1.127', 'eval_runtime': '1.71', 'eval_samples_per_second': '52.05', 'eval_steps_per_second': '13.45', 'epoch': '6'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đang sinh câu trên validation...
{
  "overall": {
    "n": 89,
    "exact_match": 0.3595505617977528,
    "unchanged_rate": 0.550561797752809,
    "keep_input_exact_match": 0.449438202247191
  },
  "by_error_type": {
    "0": {
      "n": 40,
      "exact_match": 0.7,
      "unchanged_rate": 0.7,
      "keep_input_exact_match": 1.0
    },
    "1": {
      "n": 30,
      "exact_match": 0.1,
      "unchanged_rate": 0.36666666666666664,
      "keep_input_exact_match": 0.0
    },
    "2": {
      "n": 5,
      "exact_match": 0.0,
      "unchanged_rate": 0.4,
      "keep_input_exact_match": 0.0
    },
    "3": {
      "n": 6,
      "exact_match": 0.16666666666666666,
      "unchanged_rate": 0.6666666666666666,
      "keep_input_exact_match": 0.0
    },
    "4": {
      "n": 8,
      "exact_match": 0.0,
      "unchanged_rate": 0.5,
      "keep_input_exact_match": 0.0
    }
  },
  "fold": 0,
  "eval_loss": 1.1268295049667358,
  "best_checkpoint": "/kaggle/working/bartpho_vgec/fold_0/checkpoin

In [36]:
@torch.inference_mode()
def predict_records(model, records, batch_size=2, max_length=256):
    model.eval()
    device = next(model.parameters()).device
    predictions = []
    limit = model.config.max_position_embeddings

    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]

        inputs = tokenizer(
            [row["source"] for row in batch],
            return_tensors="pt",
            padding=True,
            truncation=False,
        ).to(device)

        if inputs["input_ids"].shape[1] > limit:
            raise ValueError(f"Input vượt giới hạn {limit} tokens.")

        generated = model.generate(
            **inputs,
            num_beams=4,
            do_sample=False,
            max_length=min(max_length, limit),
            early_stopping=True,
            use_cache=True,
        )

        predictions.extend(
            tokenizer.batch_decode(
                generated,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )
        )

    return predictions

print("Đã sửa hàm inference.")

Đã sửa hàm inference.


In [37]:
cv_results = []
oof_predictions = []

for fold in range(N_FOLDS):
    run_dir = OUTPUT_DIR / f"fold_{fold}"

    required_files = [
        run_dir / "metrics.json",
        run_dir / "val_predictions.json",
        run_dir / "training_log.json",
        run_dir / "best_model" / "config.json",
    ]

    if all(path.exists() for path in required_files):
        print(f"Fold {fold}: dùng kết quả đã lưu.", flush=True)
        metrics = read_json(run_dir / "metrics.json")
    else:
        metrics = train_fold(fold)

    cv_results.append(metrics)
    oof_predictions.extend(
        read_json(run_dir / "val_predictions.json")
    )

all_records = read_json(DATA_DIR / "processed" / "all.json")

assert len(oof_predictions) == len(all_records)
assert len({row["id"] for row in oof_predictions}) == len(all_records)

oof_predictions.sort(key=lambda row: row["id"])

oof_metrics = summarize_predictions(
    oof_predictions,
    [row["prediction"] for row in oof_predictions],
)

write_json(OUTPUT_DIR / "results.json", cv_results)
write_json(OUTPUT_DIR / "oof_predictions.json", oof_predictions)
write_json(OUTPUT_DIR / "oof_metrics.json", oof_metrics)

print("\nKết quả validation tổng hợp từ 5 folds:")
print(json.dumps(oof_metrics, ensure_ascii=False, indent=2))

Fold 0: dùng kết quả đã lưu.


Loading weights:   0%|          | 0/265 [00:00<?, ?it/s]


===== FOLD 1 =====
Train: 353
Validation: 88
Độ dài lớn nhất: 64 tokens
Warmup steps: 18
Bắt đầu training...


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'loss': '19.34', 'grad_norm': '180.2', 'learning_rate': '0', 'epoch': '0.08989'}
{'loss': '16.31', 'grad_norm': '127.4', 'learning_rate': '6.667e-06', 'epoch': '0.4494'}
{'loss': '14.38', 'grad_norm': '120', 'learning_rate': '1.5e-05', 'epoch': '0.8989'}
{'eval_loss': '1.237', 'eval_runtime': '1.704', 'eval_samples_per_second': '51.65', 'eval_steps_per_second': '12.91', 'epoch': '1'}


...

{'train_runtime': '209.5', 'train_samples_per_second': '25.27', 'train_steps_per_second': '0.859', 'train_loss': '6.355', 'epoch': '8'}


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': '1.037', 'eval_runtime': '1.693', 'eval_samples_per_second': '51.98', 'eval_steps_per_second': '12.99', 'epoch': '8'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đang sinh câu trên validation...
{
  "overall": {
    "n": 88,
    "exact_match": 0.38636363636363635,
    "unchanged_rate": 0.6363636363636364,
    "keep_input_exact_match": 0.45454545454545453
  },
  "by_error_type": {
    "0": {
      "n": 40,
      "exact_match": 0.8,
      "unchanged_rate": 0.8,
      "keep_input_exact_match": 1.0
    },
    "1": {
      "n": 29,
      "exact_match": 0.06896551724137931,
      "unchanged_rate": 0.6206896551724138,
      "keep_input_exact_match": 0.0
    },
    "2": {
      "n": 5,
      "exact_match": 0.0,
      "unchanged_rate": 0.2,
      "keep_input_exact_match": 0.0
    },
    "3": {
      "n": 7,
      "exact_match": 0.0,
      "unchanged_rate": 0.2857142857142857,
      "keep_input_exact_match": 0.0
    },
    "4": {
      "n": 7,
      "exact_match": 0.0,
      "unchanged_rate": 0.42857142857142855,
      "keep_input_exact_match": 0.0
    }
  },
  "fold": 4,
  "eval_loss": 1.0371901988983154,
  "best_checkpoint": "/kaggle/working/bartpho_vg

In [44]:
%pip install -q sacrebleu rapidfuzz gec-metrics

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.0/499.0 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.7/114.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 37.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.

In [45]:
import gc
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DATA_DIR = Path("/kaggle/working/vgec_preprocessed")
OUTPUT_DIR = Path("/kaggle/working/bartpho_vgec")
METRIC_DIR = OUTPUT_DIR / "multi_metrics"
METRIC_DIR.mkdir(parents=True, exist_ok=True)

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def dump_json(path, value):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(value, f, ensure_ascii=False, indent=2)

def nfc(text):
    return unicodedata.normalize("NFC", text)

@torch.inference_mode()
def infer_validation(model, tok, records, max_length):
    predictions = []
    device = next(model.parameters()).device
    limit = model.config.max_position_embeddings

    for start in range(0, len(records), 2):
        batch = records[start:start + 2]

        inputs = tok(
            [nfc(row["source"]) for row in batch],
            padding=True,
            truncation=False,
            return_tensors="pt",
        ).to(device)

        assert inputs["input_ids"].shape[1] <= limit, (
            "Input vượt giới hạn model; không tự động cắt câu."
        )

        generated = model.generate(
            **inputs,
            num_beams=4,
            do_sample=False,
            max_length=min(max_length, limit),
            early_stopping=True,
            use_cache=True,
        )

        for sequence in generated.tolist():
            if tok.eos_token_id not in sequence[1:]:
                raise RuntimeError(
                    "Có output chưa sinh EOS. Cần tăng max_length "
                    "trước khi dùng output đó để chấm."
                )

        predictions.extend(
            tok.batch_decode(
                generated,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )
        )

    return predictions

assert torch.cuda.is_available(), "Cần bật GPU."

# Giải phóng các model inference đã load ở cell trước.
for name in ["inference_model", "val_model"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

fold_rows = {}
all_oof_rows = []

for fold in range(5):
    model_path = OUTPUT_DIR / f"fold_{fold}" / "best_model"
    fold_path = DATA_DIR / "folds" / f"fold_{fold}"

    assert (model_path / "config.json").exists(), (
        f"Chưa có model fold {fold}. Chạy cell 9 trước."
    )

    train_records = load_json(fold_path / "train.json")
    val_records = load_json(fold_path / "val.json")

    assert {
        nfc(row["source"]) for row in train_records
    }.isdisjoint({
        nfc(row["source"]) for row in val_records
    }), f"Fold {fold} bị trùng source giữa train và validation."

    config = load_json(
        OUTPUT_DIR / f"fold_{fold}" / "run_config.json"
    )

    tok = AutoTokenizer.from_pretrained(
        str(model_path),
        use_fast=False,
    )
    model = AutoModelForSeq2SeqLM.from_pretrained(
        str(model_path)
    ).to("cuda")
    model.eval()

    print(f"Đang dự đoán fold {fold}: {len(val_records)} câu...", flush=True)

    predictions = infer_validation(
        model,
        tok,
        val_records,
        config["generation_max_length"],
    )

    rows = [
        {
            **row,
            "fold": fold,
            "prediction": prediction,
        }
        for row, prediction in zip(val_records, predictions)
    ]

    fold_rows[fold] = rows
    all_oof_rows.extend(rows)

    dump_json(METRIC_DIR / f"fold_{fold}_predictions.json", rows)

    del model, tok
    gc.collect()
    torch.cuda.empty_cache()

processed = load_json(DATA_DIR / "processed" / "all.json")

assert len(all_oof_rows) == len(processed)
assert {row["id"] for row in all_oof_rows} == {
    row["id"] for row in processed
}
assert len({row["id"] for row in all_oof_rows}) == len(processed)

all_oof_rows.sort(key=lambda row: row["id"])
dump_json(METRIC_DIR / "oof_predictions.json", all_oof_rows)

print("Đã có dự đoán OOF:", len(all_oof_rows), "câu.")

Loading weights:   0%|          | 0/264 [00:00<?, ?it/s]

Đang dự đoán fold 0: 89 câu...


Loading weights:   0%|          | 0/264 [00:00<?, ?it/s]

Đang dự đoán fold 1: 88 câu...


Loading weights:   0%|          | 0/264 [00:00<?, ?it/s]

Đang dự đoán fold 2: 88 câu...


Loading weights:   0%|          | 0/264 [00:00<?, ?it/s]

Đang dự đoán fold 3: 88 câu...


Loading weights:   0%|          | 0/264 [00:00<?, ?it/s]

Đang dự đoán fold 4: 88 câu...
Đã có dự đoán OOF: 441 câu.


In [46]:
from rapidfuzz.distance import Levenshtein

TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)

def eval_tokens(text):
    return TOKEN_PATTERN.findall(nfc(text))

def tokenized_text(text):
    return " ".join(eval_tokens(text))

def extract_edits(source, target):
    src = eval_tokens(source)
    tgt = eval_tokens(target)

    edits = []
    pending = None

    for op in Levenshtein.opcodes(src, tgt):
        if op.tag == "equal":
            if pending is not None:
                a, b, c, d = pending
                edits.append((a, b, tuple(tgt[c:d])))
                pending = None
        else:
            if pending is None:
                pending = [
                    op.src_start, op.src_end,
                    op.dest_start, op.dest_end,
                ]
            else:
                pending[1] = op.src_end
                pending[3] = op.dest_end

    if pending is not None:
        a, b, c, d = pending
        edits.append((a, b, tuple(tgt[c:d])))

    return set(edits)

def prf05(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    denominator = 1.25 * tp + 0.25 * fn + fp
    f = 1.25 * tp / denominator if denominator else 0.0
    return p, r, f

def edit_counts(source, prediction, references):
    predicted = extract_edits(source, prediction)
    candidates = []

    for index, reference in enumerate(references):
        gold = extract_edits(source, reference)

        tp = len(predicted & gold)
        fp = len(predicted - gold)
        fn = len(gold - predicted)

        _, _, f = prf05(tp, fp, fn)

        # Hai bên cùng không có edit: reference phù hợp hoàn toàn.
        selection_score = 1.0 if not predicted and not gold else f

        candidates.append((
            (selection_score, tp, -fp, -fn, -index),
            (tp, fp, fn),
        ))

    return max(candidates, key=lambda item: item[0])[1]

# Kiểm tra các trường hợp cơ bản.
assert edit_counts(
    "Tôi thich tiếng anh.",
    "Tôi thích tiếng anh.",
    ["Tôi thích tiếng Anh."],
) == (1, 0, 1)

assert edit_counts(
    "Tôi thích học.",
    "Tôi rất thích học.",
    ["Tôi thích học."],
) == (0, 1, 0)

assert edit_counts(
    "Tôi thích học.",
    "Tôi thích học.",
    ["Tôi thích học."],
) == (0, 0, 0)

assert edit_counts(
    "Tôi thich học.",
    "Tôi thích học.",
    ["Tôi thích học.", "Tôi rất thích học."],
) == (1, 0, 0)

print("Đã kiểm tra các trường hợp edit cơ bản.")

Đã kiểm tra các trường hợp edit cơ bản.


In [49]:
from importlib.metadata import version
from sacrebleu.metrics import BLEU, CHRF, TER
import sys
import types
import importlib
from importlib.metadata import distribution
from importlib.machinery import ModuleSpec

# Tìm module GLEU trong thư viện đã cài,
# không chạy __init__.py vốn import cả BERTScore.
metric_module_dir = distribution("gec-metrics").locate_file(
    "gec_metrics/metrics"
)

assert (metric_module_dir / "gleu.py").is_file(), (
    f"Không tìm thấy gleu.py tại {metric_module_dir}"
)

package_name = "_vgec_isolated_metrics"

if package_name not in sys.modules:
    package = types.ModuleType(package_name)
    package.__path__ = [str(metric_module_dir)]
    package.__package__ = package_name
    package.__spec__ = ModuleSpec(
        package_name,
        loader=None,
        is_package=True,
    )
    sys.modules[package_name] = package

gleu_module = importlib.import_module(
    f"{package_name}.gleu"
)

def get_metric(metric_id):
    if metric_id == "gleuofficial":
        return gleu_module.GLEUOfficial
    raise ValueError(f"Cell này chỉ hỗ trợ gleuofficial: {metric_id}")

print("Đã load GLEUOfficial riêng, không import BERTScore.")
GLEU_ITERATIONS = 500
METRIC_SEED = 42

gleu_cls = get_metric("gleuofficial")
gleu_metric = gleu_cls(
    gleu_cls.Config(
        iter=1,
        n=4,
        unit="word",
    )
)

def gleu_multireference(sources, hypotheses, references):
    rng = np.random.default_rng(METRIC_SEED)
    scores = []

    for _ in range(GLEU_ITERATIONS):
        selected = [
            refs[int(rng.integers(len(refs)))]
            for refs in references
        ]

        score = gleu_metric.score_corpus(
            sources=sources,
            hypotheses=hypotheses,
            references=[selected],
        )
        scores.append(float(score))

    return 100.0 * float(np.mean(scores))

def score_outputs(rows, predictions):
    assert len(rows) == len(predictions) and rows

    sources_raw = [nfc(row["source"]) for row in rows]
    hypotheses_raw = [nfc(pred) for pred in predictions]

    references_raw = [
        list(dict.fromkeys(nfc(ref) for ref in row["references"]))
        for row in rows
    ]
    assert all(refs for refs in references_raw)

    tp = fp = fn = 0

    for source, hypothesis, refs in zip(
        sources_raw, hypotheses_raw, references_raw
    ):
        a, b, c = edit_counts(source, hypothesis, refs)
        tp += a
        fp += b
        fn += c

    p, r, f = prf05(tp, fp, fn)

    sources_tok = [tokenized_text(text) for text in sources_raw]
    hypotheses_tok = [tokenized_text(text) for text in hypotheses_raw]

    references_tok = [
        list(dict.fromkeys(tokenized_text(ref) for ref in refs))
        for refs in references_raw
    ]

    # SacreBLEU: shape = số reference × số câu.
    # Reference thiếu được biểu diễn bằng None.
    def reference_streams(references):
        return [
            [
                refs[index] if index < len(refs) else None
                for refs in references
            ]
            for index in range(max(map(len, references)))
        ]

    streams_tok = reference_streams(references_tok)
    streams_raw = reference_streams(references_raw)

    bleu = BLEU(
        tokenize="none",
        lowercase=False,
        effective_order=False,
    )
    chrf = CHRF(
        char_order=6,
        word_order=0,
        beta=2,
        lowercase=False,
    )
    chrfpp = CHRF(
        char_order=6,
        word_order=2,
        beta=2,
        lowercase=False,
    )
    ter = TER(
        normalized=False,
        no_punct=False,
        case_sensitive=True,
    )

    result = {
        "n": len(rows),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Edit_P": 100 * p,
        "Edit_R": 100 * r,
        "Edit_F0.5": 100 * f,
        "Exact_match": 100 * np.mean([
            hyp in refs
            for hyp, refs in zip(hypotheses_raw, references_raw)
        ]),
        "Unchanged_rate": 100 * np.mean([
            hyp == src
            for hyp, src in zip(hypotheses_raw, sources_raw)
        ]),
        "GLEU+": gleu_multireference(
            sources_tok, hypotheses_tok, references_tok
        ),
        "BLEU": bleu.corpus_score(
            hypotheses_tok, streams_tok
        ).score,
        "chrF": chrf.corpus_score(
            hypotheses_raw, streams_raw
        ).score,
        "chrF++": chrfpp.corpus_score(
            hypotheses_raw, streams_raw
        ).score,
        "TER": ter.corpus_score(
            hypotheses_tok, streams_tok
        ).score,
    }

    signatures = {
        "BLEU": str(bleu.get_signature()),
        "chrF": str(chrf.get_signature()),
        "chrF++": str(chrfpp.get_signature()),
        "TER": str(ter.get_signature()),
    }

    return result, signatures

Đã load GLEUOfficial riêng, không import BERTScore.


In [50]:
report_rows = []
saved_signatures = {}

def add_result(scope, system, rows, predictions):
    print(f"Đang chấm {scope} / {system}...", flush=True)

    metrics, signatures = score_outputs(rows, predictions)

    report_rows.append({
        "scope": scope,
        "system": system,
        **metrics,
    })

    saved_signatures[f"{scope}/{system}"] = signatures

for fold, rows in fold_rows.items():
    add_result(
        f"fold_{fold}",
        "BARTpho",
        rows,
        [row["prediction"] for row in rows],
    )

add_result(
    "OOF_all",
    "BARTpho",
    all_oof_rows,
    [row["prediction"] for row in all_oof_rows],
)

add_result(
    "OOF_all",
    "Keep_input",
    all_oof_rows,
    [row["source"] for row in all_oof_rows],
)

error_rows = [
    row for row in all_oof_rows
    if row["error_type"] != 0
]

add_result(
    "OOF_errors_only",
    "BARTpho",
    error_rows,
    [row["prediction"] for row in error_rows],
)

add_result(
    "OOF_errors_only",
    "Keep_input",
    error_rows,
    [row["source"] for row in error_rows],
)

report = pd.DataFrame(report_rows)

# Trung bình và độ lệch chuẩn giữa 5 fold.
metric_columns = [
    "Edit_P", "Edit_R", "Edit_F0.5",
    "Exact_match", "Unchanged_rate",
    "GLEU+", "BLEU", "chrF", "chrF++", "TER",
]

fold_report = report[
    report["scope"].str.startswith("fold_")
]

fold_summary = fold_report[metric_columns].agg(["mean", "std"])

print("\n===== TỪNG FOLD VÀ OOF =====")
display(report.round(3))

print("\n===== TRUNG BÌNH / ĐỘ LỆCH CHUẨN 5 FOLD =====")
display(fold_summary.round(3))

report.to_csv(
    METRIC_DIR / "metric_report.csv",
    index=False,
)
fold_summary.to_csv(
    METRIC_DIR / "fold_mean_std.csv",
)

dump_json(
    METRIC_DIR / "metric_protocol.json",
    {
        "edit_scorer": "custom Levenshtein contiguous-span exact matching",
        "official_vgec_scorer": False,
        "edit_reference_selection": (
            "best sentence F0.5; ties: TP, fewer FP, fewer FN, reference order"
        ),
        "tokenization": "Unicode NFC; regex word tokens and separate punctuation",
        "case_sensitive": True,
        "edit_zero_denominator": 0,
        "gleu": "gec-metrics gleuofficial",
        "gleu_reference_sampling": "uniform independent selection per sentence",
        "gleu_iterations": GLEU_ITERATIONS,
        "seed": METRIC_SEED,
        "score_scale": "percent / 0-100; TER can exceed 100",
        "versions": {
            package: version(package)
            for package in ["sacrebleu", "rapidfuzz", "gec-metrics"]
        },
        "signatures": saved_signatures,
    },
)

print("Đã lưu kết quả tại:", METRIC_DIR)

Đang chấm fold_0 / BARTpho...
Đang chấm fold_1 / BARTpho...
Đang chấm fold_2 / BARTpho...
Đang chấm fold_3 / BARTpho...
Đang chấm fold_4 / BARTpho...
Đang chấm OOF_all / BARTpho...


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Đang chấm OOF_all / Keep_input...


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Đang chấm OOF_errors_only / BARTpho...


...


===== TỪNG FOLD VÀ OOF =====


,scope,system,n,TP,FP,FN,Edit_P,Edit_R,Edit_F0.5,Exact_match,Unchanged_rate,GLEU+,BLEU,chrF,chrF++,TER
0,fold_0,BARTpho,89,8,34,78,19.048,9.302,15.748,35.955,55.056,67.681,79.383,84.480,84.025,20.583
1,fold_1,BARTpho,88,8,27,83,22.857,8.791,17.316,36.364,62.500,68.053,80.987,86.857,86.645,16.733
2,fold_2,BARTpho,88,6,54,93,10.000,6.061,8.850,26.136,39.773,63.981,75.270,81.835,81.226,23.980
3,fold_3,BARTpho,88,4,36,79,10.000,4.819,8.230,31.818,57.955,66.955,78.888,84.161,83.828,19.237
4,fold_4,BARTpho,88,5,28,88,15.152,5.376,11.111,38.636,63.636,67.994,80.789,87.236,86.926,18.734
5,OOF_all,BARTpho,441,31,179,421,14.762,6.858,11.997,33.787,55.782,67.255,79.453,84.929,84.544,19.860
6,OOF_all,Keep_input,441,0,0,451,0.000,0.000,0.000,45.351,100.000,65.967,81.056,90.400,90.078,16.109
7,OOF_errors_only,BARTpho,241,31,114,421,21.379,6.858,15.019,4.979,45.228,40.591,65.552,74.558,73.877,34.104
8,OOF_errors_only,Keep_input,241,0,0,451,0.000,0.000,0.000,0.000,100.000,35.150,64.392,80.698,80.026,31.785



===== TRUNG BÌNH / ĐỘ LỆCH CHUẨN 5 FOLD =====


,Edit_P,Edit_R,Edit_F0.5,Exact_match,Unchanged_rate,GLEU+,BLEU,chrF,chrF++,TER
mean,15.411,6.870,12.251,33.782,55.784,66.933,79.063,84.914,84.530,19.853
std,5.641,2.043,4.090,4.931,9.596,1.707,2.302,2.202,2.339,2.689


Đã lưu kết quả tại: /kaggle/working/bartpho_vgec/multi_metrics


In [40]:
import shutil
import gc
import torch

disk = shutil.disk_usage("/kaggle/working")
print(f"Dung lượng trống trước: {disk.free / 2**30:.2f} GB")

# Giải phóng model của lần train bị lỗi.
for name in ["final_trainer", "final_model", "full_collator", "full_dataset"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# Xóa checkpoint của lần final training bị lỗi.
checkpoint_dir = OUTPUT_DIR / "final_training_checkpoints"

if checkpoint_dir.exists():
    shutil.rmtree(checkpoint_dir)
    print("Đã xóa:", checkpoint_dir)

disk = shutil.disk_usage("/kaggle/working")
print(f"Dung lượng trống sau: {disk.free / 2**30:.2f} GB")

Dung lượng trống trước: 1.97 GB
Dung lượng trống sau: 1.97 GB


In [41]:
import statistics

best_epochs = []

for fold in range(N_FOLDS):
    log_path = OUTPUT_DIR / f"fold_{fold}" / "training_log.json"

    if log_path.exists():
        evaluations = [
            entry
            for entry in read_json(log_path)
            if "eval_loss" in entry and "epoch" in entry
        ]

        if evaluations:
            best = min(
                evaluations,
                key=lambda entry: float(entry["eval_loss"]),
            )
            best_epochs.append(float(best["epoch"]))

assert best_epochs, "Cần chạy cell 7 hoặc cell 9 trước."

FINAL_EPOCHS = max(
    1,
    math.floor(statistics.median(best_epochs) + 0.5),
)

FINAL_DIR = OUTPUT_DIR / "final_model"

set_seed(SEED)
gc.collect()
torch.cuda.empty_cache()

full_records = read_json(DATA_DIR / "processed" / "all.json")

assert full_records
assert len({row["source"] for row in full_records}) == len(full_records)
assert all(row["references"] for row in full_records)

full_dataset = CorrectionDataset(
    full_records,
    tokenizer,
    training=True,
)

final_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lengths = [
    len(source["input_ids"])
    for source in full_dataset.sources
]

lengths.extend(
    len(target)
    for refs in full_dataset.targets
    for target in refs
)

longest = max(lengths)
model_limit = final_model.config.max_position_embeddings

assert math.ceil(longest / 8) * 8 <= model_limit, (
    f"Mẫu dài nhất vượt giới hạn {model_limit} tokens."
)

FINAL_GENERATION_LENGTH = min(
    model_limit,
    max(256, math.ceil((longest + 32) / 8) * 8),
)

final_model.config.use_cache = False

full_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=final_model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

n_gpu = max(1, torch.cuda.device_count())

batches_per_epoch = math.ceil(
    len(full_records) / (TRAIN_BATCH_SIZE * n_gpu)
)

steps_per_epoch = math.ceil(
    batches_per_epoch / GRAD_ACCUMULATION
)

warmup_steps = max(
    1,
    round(0.1 * FINAL_EPOCHS * steps_per_epoch),
)

final_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR / "final_training_checkpoints"),
    num_train_epochs=FINAL_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,

    weight_decay=0.01,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,
    eval_strategy="no",
    save_strategy="no",
    load_best_model_at_end=False,

    logging_steps=5,
    logging_first_step=True,
    disable_tqdm=True,

    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    dataloader_num_workers=0,
    report_to="none",
    optim="adamw_torch",
    seed=SEED,
    data_seed=SEED,
)

final_trainer = Seq2SeqTrainer(
    model=final_model,
    args=final_args,
    train_dataset=full_dataset,
    processing_class=tokenizer,
    data_collator=full_collator,
    callbacks=[ReferenceEpochCallback(full_dataset)],
)

print("Số mẫu train:", len(full_records))
print("Epoch tốt nhất của các fold:", best_epochs)
print("Số epoch train model cuối:", FINAL_EPOCHS)
print("Bắt đầu fine-tune...", flush=True)

final_trainer.train()

final_model = final_trainer.model
final_model.gradient_checkpointing_disable()
final_model.config.use_cache = True

final_model.generation_config.max_length = FINAL_GENERATION_LENGTH
final_model.generation_config.num_beams = 4
final_model.generation_config.do_sample = False

final_trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

write_json(
    FINAL_DIR / "training_log.json",
    final_trainer.state.log_history,
)

write_json(
    FINAL_DIR / "run_config.json",
    {
        "model": MODEL_NAME,
        "seed": SEED,
        "train_size": len(full_records),
        "epochs": FINAL_EPOCHS,
        "best_epochs_from_folds": best_epochs,
        "learning_rate": LEARNING_RATE,
        "train_batch_size_per_device": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUMULATION,
        "generation_max_length": FINAL_GENERATION_LENGTH,
        "num_beams": 4,
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
    },
)

print("Đã fine-tune xong. Model lưu tại:", FINAL_DIR)

del final_trainer, final_model, full_collator, full_dataset
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/265 [00:00<?, ?it/s]

Số mẫu train: 441
Epoch tốt nhất của các fold: [3.0, 5.0, 6.0, 4.0, 5.0]
Số epoch train model cuối: 5
Bắt đầu fine-tune...


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'loss': '23.21', 'grad_norm': '198.5', 'learning_rate': '0', 'epoch': '0.07207'}
{'loss': '14.15', 'grad_norm': '137.7', 'learning_rate': '1.714e-05', 'epoch': '0.3604'}
{'loss': '13.39', 'grad_norm': '114.1', 'learning_rate': '2.905e-05', 'epoch': '0.7207'}
{'loss': '10.69', 'grad_norm': '88.86', 'learning_rate': '2.667e-05', 'epoch': '1.072'}
{'loss': '9.583', 'grad_norm': '85.06', 'learning_rate': '2.429e-05', 'epoch': '1.432'}
{'loss': '9.18', 'grad_norm': '87.24', 'learning_rate': '2.19e-05', 'epoch': '1.793'}
{'loss': '7.175', 'grad_norm': '77.5', 'learning_rate': '1.952e-05', 'epoch': '2.144'}
{'loss': '7.086', 'grad_norm': '59.87', 'learning_rate': '1.714e-05', 'epoch': '2.505'}
{'loss': '7.456', 'grad_norm': '61.99', 'learning_rate': '1.476e-05', 'epoch': '2.865'}
{'loss': '5.911', 'grad_norm': '50.81', 'learning_rate': '1.238e-05', 'epoch': '3.216'}
{'loss': '6.2', 'grad_norm': '54.74', 'learning_rate': '1e-05', 'epoch': '3.577'}
{'loss': '5.816', 'grad_norm': '55.4', 'learn

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã fine-tune xong. Model lưu tại: /kaggle/working/bartpho_vgec/final_model


In [42]:
MODEL_PATH = OUTPUT_DIR / "final_model"

# Nếu chưa chạy cell 10, dùng model fold 0:
# MODEL_PATH = OUTPUT_DIR / "fold_0" / "best_model"

assert (MODEL_PATH / "config.json").exists(), (
    f"Chưa có model tại {MODEL_PATH}"
)

if "inference_model" in globals():
    del inference_model
    gc.collect()
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_PATH),
    use_fast=False,
)

inference_model = AutoModelForSeq2SeqLM.from_pretrained(
    str(MODEL_PATH)
).to("cuda")

inference_model.eval()

config_path = MODEL_PATH / "run_config.json"

if not config_path.exists():
    config_path = MODEL_PATH.parent / "run_config.json"

generation_length = (
    read_json(config_path).get("generation_max_length", 256)
    if config_path.exists()
    else 256
)

sentences = [
    "Tôi đang học ở trường đại học.",
    "Hôm qua tôi đi đến trường để học bài bài.",
]

new_records = [
    {"source": normalize(sentence)}
    for sentence in sentences
]

predictions = predict_records(
    inference_model,
    new_records,
    batch_size=2,
    max_length=generation_length,
)

for sentence, prediction in zip(sentences, predictions):
    print("Input :", sentence)
    print("Output:", prediction)
    print()

Loading weights:   0%|          | 0/264 [00:00<?, ?it/s]

Input : Tôi đang học ở trường đại học.
Output: Tôi đang học ở trường đại học.

Input : Hôm qua tôi đi đến trường để học bài bài.
Output: Hôm qua tôi đi đến trường để học bài.



In [43]:
import shutil

zip_path = shutil.make_archive(
    str(OUTPUT_DIR / "vgec_finetuned_model"),
    "zip",
    root_dir=MODEL_PATH,
)

print("Đã tạo ZIP:", zip_path)

Đã tạo ZIP: /kaggle/working/bartpho_vgec/vgec_finetuned_model.zip
